# Experimento A, segunda versão: com × sem tratamento de outliers
Este notebook repete a avaliação do notebook `imputacoes_mesma_prova_tratados` (os 3 métodos na mesma prova, com o KNN padronizado) na base **não tratada**: as médias mensais da base limpa, com os outliers. Depois, compara as duas bases. É a mesma comparação que o notebook `imputacoes_outliers_nao_tratados` faz para o A original.
- **Mesma prova nas duas bases:** o sorteio fica só nos 118 meses completos da tratada, e as células sorteadas são as mesmas nas duas bases. A diferença de erro vem só dos 8 meses que têm valores extremos.
- **O mesmo KNN nas duas bases:** o k e os pesos escolhidos na tratada (lidos do Excel dela). Uma seção à parte confere se, na não tratada, a escolha seria a mesma.
- **Células que mudam:** além das métricas, em quantas células o valor imputado é o mesmo nas duas bases e, quando ele muda, quanto muda.

# Set Up

In [1]:
import numpy as np
import pandas as pd
from pcj.utils import dados, BASE_DIR
from pcj.processamento import decompor_estatisticas
from pcj.metricas_avaliacao import avaliacao_mesma_prova, metricas_juntando

In [2]:
# Base não tratada: as médias mensais da base limpa, com os outliers (como em aplicacoes/outliers)
df, metadados, variaveis = dados(r"data\processed\dados_limpos.xlsx")
df_med = decompor_estatisticas(df, estatistica="med")

# Base tratada: para saber quais meses completos ela tem (ver "Mesma prova") e,
# em "Células que mudam", para rodar a avaliação dela de novo
df_tratada, _, _ = dados(r"data\processed\dados_med_tratados.xlsx")

# Resultados da tratada, gravados pelo notebook imputacoes_mesma_prova_tratados
# (se algo mudar lá, rode aquele antes deste)
caminho_tratada = BASE_DIR / "data" / "output" / "imputacoes_mesma_prova_tratados.xlsx"

# Taxas de falhas e métodos (os padrões da avaliacao_mesma_prova)
taxas = [0.05, 0.1, 0.2, 0.3]
metodos = ["media", "mediana", "knn"]

# Mesma prova
A base não tratada tem mais meses completos que a tratada: os meses que tiveram valores tratados ficaram com NaN e saem do `dropna()`. Para comparar as duas bases com a mesma prova, o sorteio fica só nos meses completos da tratada (`meses_sorteio`). Os outros meses completos da não tratada continuam na base que o imputador vê (e na padronização do KNN), mas não são sorteados. Com as mesmas seeds, as células sorteadas são as mesmas do notebook `imputacoes_mesma_prova_tratados`.

In [3]:
# Meses completos (sem NaN) de cada base
completos = df_med.dropna()
completos_tratada = df_tratada.dropna()
print("Meses completos, não tratada:", len(completos))
print("Meses completos, tratada:", len(completos_tratada))

# Os índices das duas bases são diferentes (na tratada, lida do Excel, vão de 0 a 190),
# por isso os meses são casados pela data. isin marca True nas datas que estão na outra tabela
na_tratada = completos["Data"].isin(completos_tratada["Data"])
meses_sorteio = completos.index[na_tratada]
print("Meses que podem ser sorteados:", len(meses_sorteio))

# Meses completos só na não tratada (o ~ inverte True e False): ficam na base, mas não são sorteados
completos.loc[~na_tratada, "Data"].dt.strftime("%Y-%m").tolist()

Meses completos, não tratada: 126
Meses completos, tratada: 118
Meses que podem ser sorteados: 118


['2009-12',
 '2014-02',
 '2014-07',
 '2014-09',
 '2014-11',
 '2015-11',
 '2016-02',
 '2022-10']

# Conferência da escolha do k
A avaliação usa o k e os pesos escolhidos na tratada: o mesmo método nas duas bases, para que a diferença venha só dos meses extremos. Nesta seção, a mesma grade da tratada (k = 1, 3 … 15 × pesos; seeds 100 a 129; sorteio nos 118 meses da mesma prova) roda na não tratada, só para ver se a escolha seria a mesma. Ela não muda a comparação.

In [4]:
# Escolha feita na tratada: a mesma conta do notebook dela (maior R² médio na média das 4 taxas)
escolha_k_tratada = pd.read_excel(caminho_tratada, sheet_name="escolha_k")
tabela_escolha_tratada = escolha_k_tratada.pivot(index=["k", "pesos"], columns="pct_faltantes", values="r2_medio")
tabela_escolha_tratada["media_taxas"] = tabela_escolha_tratada.mean(axis=1)

# idxmax devolve o rótulo da linha com o maior valor: aqui, o par (k, pesos)
k_escolhido, pesos_escolhidos = tabela_escolha_tratada["media_taxas"].idxmax()
print("Escolhido na tratada (usado na avaliação): k =", k_escolhido, "| pesos =", pesos_escolhidos)

Escolhido na tratada (usado na avaliação): k = 13 | pesos = distance


In [5]:
grade_k = [1, 3, 5, 7, 9, 11, 13, 15]
grade_pesos = ["uniform", "distance"]
linhas = []

for k in grade_k:
    for pesos in grade_pesos:
        # Só o KNN, nas seeds 100 a 129 (outras células, não as da avaliação), com a mesma prova.
        # O _ recebe a tabela por rodada, que aqui não é usada
        _, celulas_k = avaliacao_mesma_prova(df_med, seeds=range(100, 130), metodos=("knn",),
                                             n_neighbors=k, weights=pesos, devolver_celulas=True,
                                             meses_sorteio=meses_sorteio)
        juntando_k = metricas_juntando(celulas_k)

        # R² médio entre as variáveis, em cada taxa (cada grupo = as 16 variáveis de uma taxa)
        for taxa, grupo in juntando_k.groupby("pct_faltantes"):
            linhas.append({
                "k": k,
                "pesos": pesos,
                "pct_faltantes": taxa,
                "r2_medio": grupo["r2"].mean()
            })

escolha_k = pd.DataFrame(linhas)

Uma linha por k e pesos; uma coluna por taxa de falhas (R² médio entre as variáveis, na não tratada), a média das 4 taxas e, na última coluna, a mesma média na tratada.

In [6]:
# pivot: uma linha por (k, pesos), uma coluna por taxa
tabela_escolha = escolha_k.pivot(index=["k", "pesos"], columns="pct_faltantes", values="r2_medio")

# Média das 4 taxas (mean(axis=1) tira a média de cada linha)
tabela_escolha["media_taxas"] = tabela_escolha.mean(axis=1)

k_nao_tratada, pesos_nao_tratada = tabela_escolha["media_taxas"].idxmax()
print("Escolha na não tratada: k =", k_nao_tratada, "| pesos =", pesos_nao_tratada)
print("Escolha na tratada: k =", k_escolhido, "| pesos =", pesos_escolhidos)

# O melhor de cada taxa, para ver se a escolha mudaria com a taxa
for taxa in taxas:
    k_taxa, pesos_taxa = tabela_escolha[taxa].idxmax()
    print(f"  melhor a {taxa:.0%} na não tratada: k = {k_taxa}, pesos = {pesos_taxa}")

# Coluna da tratada: o pandas preenche pelo índice (k, pesos), linha a linha
tabela_escolha["media_taxas_tratada"] = tabela_escolha_tratada["media_taxas"]
tabela_escolha.round(3)

Escolha na não tratada: k = 13 | pesos = distance
Escolha na tratada: k = 13 | pesos = distance
  melhor a 5% na não tratada: k = 15, pesos = distance
  melhor a 10% na não tratada: k = 11, pesos = distance
  melhor a 20% na não tratada: k = 13, pesos = distance
  melhor a 30% na não tratada: k = 13, pesos = distance


pct_faltantes   0.05    0.1    0.2    0.3  media_taxas  media_taxas_tratada
k  pesos                                                                   
1  distance   -0.503 -0.264 -0.304 -0.369       -0.360               -0.152
   uniform    -0.503 -0.264 -0.304 -0.369       -0.360               -0.152
3  distance    0.181  0.214  0.172  0.118        0.171                0.218
   uniform     0.172  0.202  0.161  0.117        0.163                0.208
5  distance    0.275  0.288  0.248  0.203        0.254                0.271
   uniform     0.262  0.273  0.233  0.197        0.241                0.256
7  distance    0.308  0.312  0.276  0.236        0.283                0.295
   uniform     0.292  0.293  0.257  0.225        0.267                0.278
9  distance    0.327  0.318  0.288  0.254        0.297                0.301
   uniform     0.311  0.296  0.266  0.239        0.278                0.281
11 distance    0.332  0.330  0.295  0.259        0.304                0.305
   uniform     0.314  0.308  0.272  0.241        0.284                0.283
13 distance    0.336  0.330  0.296  0.262        0.306                0.307
   uniform     0.317  0.307  0.270  0.241        0.284                0.284
15 distance    0.336  0.330  0.296  0.262        0.306                0.305
   uniform     0.317  0.305  0.269  0.237        0.282                0.280

# Avaliação
30 rodadas (seeds 0 a 29) × 4 taxas, com os 3 métodos nas mesmas células, o KNN com o k e os pesos da tratada e o sorteio nos 118 meses da mesma prova.

In [7]:
# tabela: uma linha por seed × taxa × método × variável (as métricas de cada rodada)
# celulas: uma linha por célula avaliada (valor verdadeiro e imputado), usada para juntar as rodadas
tabela, celulas = avaliacao_mesma_prova(df_med, n_neighbors=int(k_escolhido), weights=pesos_escolhidos,
                                        devolver_celulas=True, meses_sorteio=meses_sorteio)
tabela.head()

,variavel,rmse,mae,r2,n_celulas,metodo,pct_faltantes,seed
0,COR,69.716523,65.687603,-0.000855,5,media,0.05,0
1,TURB.,53.440060,46.449587,-0.010502,5,media,0.05,0
2,pH,0.141906,0.115207,-0.936277,5,media,0.05,0
3,ALC.,16.653125,15.920661,-1.602539,5,media,0.05,0
4,AC.,3.635749,2.775207,-0.197344,5,media,0.05,0


Conferência: número de rodadas, células fora dos meses do sorteio (precisa ser 0) e células avaliadas por variável em cada taxa (as mesmas da tratada: 5, 11, 23 e 35)

In [8]:
print("Rodadas:", tabela["seed"].nunique())

# A coluna indice guarda o rótulo da linha de cada célula sorteada
print("Células fora dos meses do sorteio:", (~celulas["indice"].isin(meses_sorteio)).sum())

# Células por variável em cada rodada: o número é o mesmo em todas as rodadas e métodos,
# por isso basta a primeira linha de cada método e taxa (first)
tabela.groupby(["pct_faltantes", "metodo"], sort=False)["n_celulas"].first().unstack()

Rodadas: 30
Células fora dos meses do sorteio: 0


metodo,media,mediana,knn
pct_faltantes,,,
0.05,5,5,5
0.10,11,11,11
0.20,23,23,23
0.30,35,35,35


# Métricas juntando as rodadas
Como na tratada: RMSE, MAE e R² calculados uma vez sobre todas as células de cada método × taxa × variável (`metricas_juntando`), e o DP do RMSE e do MAE entre as 30 rodadas.

In [9]:
juntando = metricas_juntando(celulas)

# DP entre as rodadas: o groupby separa a tabela por método, taxa e variável; o std calcula o DP das 30 rodadas
desvio = tabela.groupby(["metodo", "pct_faltantes", "variavel"], sort=False)[["rmse", "mae"]].std()

# add_suffix renomeia as colunas (rmse -> rmse_dp, mae -> mae_dp); reset_index devolve as chaves do groupby às colunas
desvio = desvio.add_suffix("_dp").reset_index()

# merge junta as duas tabelas pelas linhas com o mesmo método, a mesma taxa e a mesma variável
resumo = juntando.merge(desvio, on=["metodo", "pct_faltantes", "variavel"])
resumo = resumo[["metodo", "pct_faltantes", "variavel", "rmse", "rmse_dp", "mae", "mae_dp",
                 "r2", "n_celulas", "n_seeds"]]
print("Linhas:", len(resumo), "(3 métodos × 4 taxas × 16 variáveis)")
resumo.head()

Linhas: 192 (3 métodos × 4 taxas × 16 variáveis)


,metodo,pct_faltantes,variavel,rmse,rmse_dp,mae,mae_dp,r2,n_celulas,n_seeds
0,media,0.05,COR,85.985252,30.694843,68.989642,22.668821,-0.037296,150,30
1,media,0.05,TURB.,70.764762,23.185279,57.400551,16.646832,-0.011247,150,30
2,media,0.05,pH,0.152779,0.050763,0.122854,0.043294,-0.020539,150,30
3,media,0.05,ALC.,19.482914,5.029581,16.739063,4.683499,-0.080192,150,30
4,media,0.05,AC.,5.180481,2.237031,3.812562,1.685738,-0.014401,150,30


Conferência com o notebook `imputacoes_outliers_nao_tratados` (A original, na mesma base): a 5%, a média e a mediana imputam as mesmas células (as mesmas seeds e os mesmos meses do sorteio) do mesmo jeito, e a diferença precisa ser 0 (só sobra o arredondamento da gravação no Excel). O KNN fica de fora: lá, ele é o antigo, a 30%.

In [10]:
caminho_A = BASE_DIR / "data" / "output" / "imputacoes_outliers_nao_tratados.xlsx"
resumo_A = pd.read_excel(caminho_A, sheet_name="resumo")

# Só a média e a mediana (no A, as duas a 5%). isin marca True nas linhas desses métodos
resumo_A = resumo_A[resumo_A["metodo"].isin(["media", "mediana"])]

# merge junta as linhas do A e do novo com o mesmo método, a mesma taxa e a mesma variável.
# suffixes diferencia as colunas de mesmo nome nas duas tabelas (rmse_A, rmse_novo...)
conferencia_A = resumo_A.merge(resumo, on=["metodo", "pct_faltantes", "variavel"], suffixes=("_A", "_novo"))

maior_diferenca = 0
for metrica in ["rmse", "mae", "r2"]:
    diferenca = (conferencia_A[f"{metrica}_novo"] - conferencia_A[f"{metrica}_A"]).abs().max()
    maior_diferenca = max(maior_diferenca, diferenca)
print("Média e mediana a 5%: linhas", len(conferencia_A), "| maior diferença absoluta:", maior_diferenca)

Média e mediana a 5%: linhas 32 | maior diferença absoluta: 7.275957614183426e-12


# Comparação: tratada × não tratada
Os resultados da tratada vêm do Excel do notebook `imputacoes_mesma_prova_tratados`. Diferença = tratada − não tratada:
- RMSE e MAE: diferença negativa = a tratada errou menos.
- R²: o contrário; diferença positiva = a tratada foi melhor.

## Visão geral
R² médio entre as 16 variáveis, por método e taxa, nas duas bases.

In [11]:
resumo_tratada = pd.read_excel(caminho_tratada, sheet_name="resumo")
rodadas_tratada = pd.read_excel(caminho_tratada, sheet_name="rodadas")

linhas = []
for metodo in metodos:
    for taxa in taxas:
        # Linhas deste método e desta taxa (as 16 variáveis) em cada base
        da_tratada = resumo_tratada[(resumo_tratada["metodo"] == metodo) & (resumo_tratada["pct_faltantes"] == taxa)]
        da_nao_tratada = resumo[(resumo["metodo"] == metodo) & (resumo["pct_faltantes"] == taxa)]

        linhas.append({
            "metodo": metodo,
            "pct_faltantes": taxa,
            "r2_medio_tratada": da_tratada["r2"].mean(),
            "r2_medio_nao_tratada": da_nao_tratada["r2"].mean(),
            "diferenca": da_tratada["r2"].mean() - da_nao_tratada["r2"].mean()
        })

pd.DataFrame(linhas).round(3)

,metodo,pct_faltantes,r2_medio_tratada,r2_medio_nao_tratada,diferenca
0,media,0.05,-0.024,-0.040,0.016
1,media,0.10,-0.019,-0.032,0.013
2,media,0.20,-0.019,-0.036,0.017
3,media,0.30,-0.022,-0.042,0.020
4,mediana,0.05,-0.099,-0.098,-0.001
5,mediana,0.10,-0.091,-0.091,0.000
6,mediana,0.20,-0.097,-0.089,-0.008
7,mediana,0.30,-0.099,-0.089,-0.010
8,knn,0.05,0.407,0.399,0.008
9,knn,0.10,0.349,0.346,0.003


## Por taxa e variável
Para cada taxa, uma tabela por métrica: variáveis nas linhas; para cada método, o valor na tratada, na não tratada e a diferença.

In [12]:
linhas = []

for taxa in taxas:
    # Linhas desta taxa em cada base
    resumo_tratada_taxa = resumo_tratada[resumo_tratada["pct_faltantes"] == taxa]
    resumo_taxa = resumo[resumo["pct_faltantes"] == taxa]

    for metrica in ["rmse", "mae", "r2"]:
        # pivot reorganiza cada resumo: variáveis nas linhas, métodos nas colunas
        tratada = resumo_tratada_taxa.pivot(index="variavel", columns="metodo", values=metrica)
        nao_tratada = resumo_taxa.pivot(index="variavel", columns="metodo", values=metrica)

        # Variáveis na ordem da base. As colunas são preenchidas pelo índice (nome da variável)
        tabela_metrica = pd.DataFrame(index=variaveis)
        for metodo in metodos:
            tabela_metrica[f"{metodo} tratada"] = tratada[metodo]
            tabela_metrica[f"{metodo} não tratada"] = nao_tratada[metodo]
            tabela_metrica[f"{metodo} diferença"] = tratada[metodo] - nao_tratada[metodo]

            # Versão longa, para gravar no Excel: uma linha por taxa × métrica × método × variável
            for variavel in variaveis:
                linhas.append({
                    "pct_faltantes": taxa,
                    "metrica": metrica,
                    "metodo": metodo,
                    "variavel": variavel,
                    "tratada": tratada.at[variavel, metodo],
                    "nao_tratada": nao_tratada.at[variavel, metodo],
                    "diferenca": tratada.at[variavel, metodo] - nao_tratada.at[variavel, metodo]
                })

        # display mostra a tabela formatada, como a última linha de uma célula, mesmo dentro do laço
        print(f"{metrica.upper()}, {taxa:.0%} de falhas")
        display(tabela_metrica.round(3))

comparacao = pd.DataFrame(linhas)

RMSE, 5% de falhas


,media tratada,media não tratada,media diferença,mediana tratada,mediana não tratada,mediana diferença,knn tratada,knn não tratada,knn diferença
COR,85.262,85.985,-0.723,92.787,91.989,0.798,44.595,43.960,0.635
TURB.,70.938,70.765,0.173,80.094,79.618,0.475,45.572,44.271,1.301
pH,0.153,0.153,0.001,0.158,0.166,-0.008,0.129,0.132,-0.003
ALC.,18.989,19.483,-0.494,18.799,18.790,0.010,9.588,9.548,0.039
AC.,5.191,5.180,0.011,5.497,5.497,0.000,4.228,4.308,-0.081
O.C.,1.830,1.878,-0.047,1.831,1.832,-0.001,1.622,1.668,-0.046
O.D.,0.822,0.822,-0.001,0.821,0.821,0.000,0.649,0.647,0.002
Cl,13.829,14.358,-0.530,13.781,13.781,-0.000,8.567,8.589,-0.022
DUR.,14.833,14.856,-0.023,15.195,15.032,0.162,11.091,11.032,0.058
Fe,1.655,1.664,-0.009,1.768,1.754,0.014,1.012,1.040,-0.028


MAE, 5% de falhas


,media tratada,media não tratada,media diferença,mediana tratada,mediana não tratada,mediana diferença,knn tratada,knn não tratada,knn diferença
COR,67.399,68.990,-1.591,63.867,63.813,0.053,30.361,29.270,1.091
TURB.,56.683,57.401,-0.717,53.620,53.467,0.153,29.698,28.312,1.387
pH,0.122,0.123,-0.001,0.119,0.129,-0.009,0.099,0.100,-0.001
ALC.,16.134,16.739,-0.605,15.580,15.553,0.027,7.299,7.306,-0.007
AC.,3.643,3.813,-0.169,3.460,3.460,0.000,2.888,3.024,-0.136
O.C.,1.467,1.514,-0.046,1.465,1.471,-0.006,1.327,1.322,0.005
O.D.,0.653,0.653,-0.000,0.649,0.649,0.000,0.492,0.487,0.005
Cl,10.441,11.532,-1.092,10.187,10.187,0.000,5.902,5.937,-0.035
DUR.,12.103,12.275,-0.173,12.213,12.360,-0.147,8.055,8.047,0.008
Fe,1.330,1.354,-0.024,1.219,1.220,-0.001,0.661,0.679,-0.018


R2, 5% de falhas


,media tratada,media não tratada,media diferença,mediana tratada,mediana não tratada,mediana diferença,knn tratada,knn não tratada,knn diferença
COR,-0.020,-0.037,0.017,-0.208,-0.187,-0.021,0.721,0.729,-0.008
TURB.,-0.016,-0.011,-0.005,-0.295,-0.280,-0.015,0.581,0.604,-0.024
pH,-0.028,-0.021,-0.007,-0.087,-0.198,0.111,0.271,0.233,0.038
ALC.,-0.026,-0.080,0.054,-0.006,-0.005,-0.001,0.738,0.741,-0.002
AC.,-0.019,-0.014,-0.004,-0.142,-0.142,0.000,0.324,0.298,0.026
O.C.,-0.030,-0.084,0.054,-0.031,-0.032,0.001,0.191,0.145,0.046
O.D.,-0.043,-0.045,0.001,-0.043,-0.043,0.000,0.349,0.352,-0.004
Cl,-0.019,-0.098,0.080,-0.012,-0.012,0.000,0.609,0.607,0.002
DUR.,-0.019,-0.022,0.003,-0.069,-0.046,-0.023,0.430,0.436,-0.006
Fe,-0.027,-0.038,0.011,-0.172,-0.153,-0.019,0.616,0.594,0.021


... (14 outputs left out)

MAE, 30% de falhas


,media tratada,media não tratada,media diferença,mediana tratada,mediana não tratada,mediana diferença,knn tratada,knn não tratada,knn diferença
COR,73.639,75.876,-2.237,68.420,68.322,0.098,41.629,41.326,0.303
TURB.,55.776,56.956,-1.180,51.994,51.535,0.459,35.777,35.483,0.294
pH,0.126,0.127,-0.001,0.129,0.132,-0.003,0.110,0.111,-0.001
ALC.,15.593,16.079,-0.486,15.458,15.545,-0.087,9.441,9.538,-0.097
AC.,3.476,3.811,-0.335,3.245,3.245,0.000,3.305,3.371,-0.066
O.C.,1.666,1.694,-0.027,1.681,1.669,0.012,1.584,1.568,0.015
O.D.,0.668,0.667,0.001,0.666,0.671,-0.006,0.548,0.548,-0.000
Cl,10.385,11.755,-1.371,10.286,10.276,0.010,7.113,7.065,0.048
DUR.,12.032,12.445,-0.413,12.040,12.096,-0.056,9.143,9.020,0.123
Fe,1.425,1.465,-0.040,1.302,1.304,-0.002,0.845,0.842,0.003


R2, 30% de falhas


,media tratada,media não tratada,media diferença,mediana tratada,mediana não tratada,mediana diferença,knn tratada,knn não tratada,knn diferença
COR,-0.028,-0.032,0.004,-0.194,-0.169,-0.026,0.457,0.467,-0.011
TURB.,-0.025,-0.020,-0.005,-0.286,-0.247,-0.039,0.409,0.421,-0.012
pH,-0.027,-0.030,0.004,-0.120,-0.174,0.053,0.170,0.156,0.013
ALC.,-0.023,-0.062,0.039,-0.040,-0.041,0.001,0.545,0.554,-0.009
AC.,-0.026,-0.042,0.016,-0.107,-0.107,0.000,0.042,0.030,0.011
O.C.,-0.019,-0.033,0.014,-0.031,-0.013,-0.018,0.016,0.023,-0.007
O.D.,-0.028,-0.026,-0.003,-0.022,-0.038,0.016,0.242,0.247,-0.006
Cl,-0.011,-0.142,0.131,-0.024,-0.015,-0.009,0.455,0.474,-0.019
DUR.,-0.027,-0.065,0.038,-0.059,-0.035,-0.025,0.293,0.312,-0.019
Fe,-0.017,-0.021,0.004,-0.162,-0.134,-0.028,0.497,0.504,-0.007


# Rodadas ganhas
Com a mesma prova, cada rodada compara as duas bases nas mesmas células. Para cada taxa, método e variável: em quantas das 30 rodadas a tratada errou menos, em quantas empataram e em quantas a não tratada errou menos. Só RMSE e MAE: o R² de uma rodada oscila demais (é o motivo de juntar as rodadas).

In [13]:
# merge junta as rodadas das duas bases pela mesma seed, taxa, método e variável.
# suffixes diferencia as colunas que têm o mesmo nome nas duas tabelas (rmse_tratada, rmse_nao_tratada...)
pares = rodadas_tratada.merge(tabela, on=["seed", "pct_faltantes", "metodo", "variavel"],
                              suffixes=("_tratada", "_nao_tratada"))
print("Pares (seed × taxa × método × variável):", len(pares))

linhas = []

# Cada grupo = as 30 rodadas de uma taxa, de um método e de uma variável
for (taxa, metodo, variavel), grupo in pares.groupby(["pct_faltantes", "metodo", "variavel"], sort=False):
    for metrica in ["rmse", "mae"]:
        erro_tratada = grupo[f"{metrica}_tratada"]
        erro_nao_tratada = grupo[f"{metrica}_nao_tratada"]

        # Empate: o mesmo erro nas duas bases. np.isclose ignora diferenças minúsculas de
        # arredondamento (os números da tratada passaram pelo Excel)
        empate = np.isclose(erro_tratada, erro_nao_tratada)

        linhas.append({
            "pct_faltantes": taxa,
            "metrica": metrica,
            "metodo": metodo,
            "variavel": variavel,
            "tratada_menor": ((erro_tratada < erro_nao_tratada) & ~empate).sum(),
            "empate": empate.sum(),
            "nao_tratada_menor": ((erro_tratada > erro_nao_tratada) & ~empate).sum()
        })

rodadas_ganhas = pd.DataFrame(linhas)

Pares (seed × taxa × método × variável): 5760


Visão geral: a soma das 16 variáveis (16 × 30 = 480 comparações) por método e taxa.

In [14]:
for metrica in ["rmse", "mae"]:
    parte = rodadas_ganhas[rodadas_ganhas["metrica"] == metrica]

    # groupby + sum: soma as contagens das 16 variáveis de cada método e taxa
    total = parte.groupby(["metodo", "pct_faltantes"])[["tratada_menor", "empate", "nao_tratada_menor"]].sum()

    # Métodos na ordem média, mediana, KNN (o groupby os põe em ordem alfabética)
    total = total.loc[metodos]

    print(metrica.upper(), "(rodadas, soma das 16 variáveis)")
    display(total)

RMSE (rodadas, soma das 16 variáveis)


tratada_menor  empate  nao_tratada_menor
metodo  pct_faltantes                                          
media   0.05                     292       0                188
        0.10                     282       0                198
        0.20                     301       0                179
        0.30                     303       1                176
mediana 0.05                      96     197                187
        0.10                      93     192                195
        0.20                      87     189                204
        0.30                      96     172                212
knn     0.05                     239       1                240
        0.10                     253       0                227
        0.20                     240       0                240
        0.30                     256       0                224

MAE (rodadas, soma das 16 variáveis)


tratada_menor  empate  nao_tratada_menor
metodo  pct_faltantes                                          
media   0.05                     334       0                146
        0.10                     367       0                113
        0.20                     393       0                 87
        0.30                     402       0                 78
mediana 0.05                     130     196                154
        0.10                     143     189                148
        0.20                     137     188                155
        0.30                     150     173                157
knn     0.05                     248       0                232
        0.10                     244       0                236
        0.20                     247       0                233
        0.30                     254       0                226

Por variável: para cada taxa, uma tabela por métrica (como no notebook `imputacoes_outliers_nao_tratados`): variáveis nas linhas; para cada método, as três contagens (somam 30). A última linha soma as 16 variáveis.

In [15]:
for taxa in taxas:
    for metrica in ["rmse", "mae"]:
        parte = rodadas_ganhas[(rodadas_ganhas["pct_faltantes"] == taxa) & (rodadas_ganhas["metrica"] == metrica)]

        tabela_ganhas = pd.DataFrame(index=variaveis)
        for metodo in metodos:
            # Linhas deste método, com a variável como índice, para preencher a tabela pela variável
            do_metodo = parte[parte["metodo"] == metodo].set_index("variavel")
            tabela_ganhas[f"{metodo}: tratada menor"] = do_metodo["tratada_menor"]
            tabela_ganhas[f"{metodo}: empate"] = do_metodo["empate"]
            tabela_ganhas[f"{metodo}: não tratada menor"] = do_metodo["nao_tratada_menor"]

        # Linha nova "Total": a soma de cada coluna
        tabela_ganhas.loc["Total"] = tabela_ganhas.sum()

        print(f"{metrica.upper()}, {taxa:.0%} de falhas (rodadas)")
        display(tabela_ganhas)

RMSE, 5% de falhas (rodadas)


,media: tratada menor,media: empate,media: não tratada menor,mediana: tratada menor,mediana: empate,mediana: não tratada menor,knn: tratada menor,knn: empate,knn: não tratada menor
COR,18,0,12,5,0,25,11,0,19
TURB.,15,0,15,0,3,27,11,0,19
pH,14,0,16,4,26,0,19,0,11
ALC.,21,0,9,1,26,3,14,0,16
AC.,15,0,15,0,30,0,17,0,13
O.C.,20,0,10,15,1,14,16,0,14
O.D.,20,0,10,0,30,0,14,0,16
Cl,21,0,9,0,30,0,15,0,15
DUR.,15,0,15,12,0,18,16,0,14
Fe,19,0,11,4,7,19,15,1,14


MAE, 5% de falhas (rodadas)


,media: tratada menor,media: empate,media: não tratada menor,mediana: tratada menor,mediana: empate,mediana: não tratada menor,knn: tratada menor,knn: empate,knn: não tratada menor
COR,22,0,8,14,0,16,12,0,18
TURB.,23,0,7,7,3,20,9,0,21
pH,19,0,11,4,26,0,17,0,13
ALC.,19,0,11,1,26,3,16,0,14
AC.,20,0,10,0,30,0,21,0,9
O.C.,18,0,12,17,0,13,11,0,19
O.D.,18,0,12,0,30,0,14,0,16
Cl,21,0,9,0,30,0,16,0,14
DUR.,16,0,14,14,0,16,16,0,14
Fe,20,0,10,11,7,12,16,0,14


RMSE, 10% de falhas (rodadas)


,media: tratada menor,media: empate,media: não tratada menor,mediana: tratada menor,mediana: empate,mediana: não tratada menor,knn: tratada menor,knn: empate,knn: não tratada menor
COR,18,0,12,1,0,29,11,0,19
TURB.,19,0,11,0,8,22,9,0,21
pH,14,0,16,8,22,0,18,0,12
ALC.,18,0,12,5,21,4,20,0,10
AC.,17,0,13,0,30,0,15,0,15
O.C.,19,0,11,10,2,18,11,0,19
O.D.,15,0,15,5,23,2,19,0,11
Cl,21,0,9,1,29,0,14,0,16
DUR.,20,0,10,14,0,16,17,0,13
Fe,18,0,12,3,5,22,19,0,11


... (6 outputs left out)

RMSE, 30% de falhas (rodadas)


,media: tratada menor,media: empate,media: não tratada menor,mediana: tratada menor,mediana: empate,mediana: não tratada menor,knn: tratada menor,knn: empate,knn: não tratada menor
COR,19,0,11,0,0,30,14,0,16
TURB.,13,0,17,0,10,20,14,0,16
pH,19,0,11,9,21,0,20,0,10
ALC.,20,0,10,8,17,5,15,0,15
AC.,19,0,11,0,30,0,18,0,12
O.C.,16,0,14,11,0,19,12,0,18
O.D.,14,1,15,8,20,2,15,0,15
Cl,25,0,5,3,23,4,15,0,15
DUR.,23,0,7,12,2,16,9,0,21
Fe,20,0,10,0,6,24,15,0,15


MAE, 30% de falhas (rodadas)


,media: tratada menor,media: empate,media: não tratada menor,mediana: tratada menor,mediana: empate,mediana: não tratada menor,knn: tratada menor,knn: empate,knn: não tratada menor
COR,27,0,3,14,0,16,15,0,15
TURB.,27,0,3,1,11,18,9,0,21
pH,19,0,11,9,21,0,18,0,12
ALC.,23,0,7,10,17,3,20,0,10
AC.,28,0,2,0,30,0,18,0,12
O.C.,18,0,12,11,0,19,11,0,19
O.D.,12,0,18,8,20,2,12,0,18
Cl,30,0,0,5,23,2,14,0,16
DUR.,24,0,6,19,2,9,11,0,19
Fe,30,0,0,13,6,11,14,0,16


# Células que mudam
O Excel da tratada guarda só as métricas, não o valor imputado em cada célula. Por isso, esta seção roda a avaliação da tratada de novo (a mesma função, com o mesmo k e os mesmos pesos; leva poucos segundos) e confere que as métricas são as do Excel.

Depois, junta as células das duas bases: a mesma seed, taxa, método, variável e mês. Os meses são casados pela data, porque os índices das duas bases são diferentes. Para cada célula:
- **igual:** o valor imputado é o mesmo nas duas bases (`np.isclose`, que ignora diferenças minúsculas de arredondamento);
- **mudança em DP:** a diferença entre os dois valores imputados (sem o sinal), dividida pelo desvio padrão da variável nos 118 meses da prova (os mesmos valores nas duas bases). Assim, a régua é a mesma para todas as variáveis e métodos: 0,1 = o valor imputado se moveu um décimo do DP da variável.

In [16]:
# A avaliação da tratada de novo, agora com as células (valor verdadeiro e imputado de cada uma)
tabela_tratada, celulas_tratada = avaliacao_mesma_prova(df_tratada, n_neighbors=int(k_escolhido),
                                                        weights=pesos_escolhidos, devolver_celulas=True)

# Conferência: as métricas juntando as rodadas são as do Excel da tratada
# (a maior diferença, em valor absoluto, precisa ser ~0; só sobra o arredondamento da gravação no Excel)
juntando_tratada = metricas_juntando(celulas_tratada)
conferencia = resumo_tratada.merge(juntando_tratada, on=["metodo", "pct_faltantes", "variavel"],
                                   suffixes=("_excel", "_de_novo"))

maior_diferenca = 0
for metrica in ["rmse", "mae", "r2"]:
    diferenca = (conferencia[f"{metrica}_excel"] - conferencia[f"{metrica}_de_novo"]).abs().max()
    maior_diferenca = max(maior_diferenca, diferenca)
print("Tratada de novo × Excel: linhas", len(conferencia), "| maior diferença absoluta:", maior_diferenca)

Tratada de novo × Excel: linhas 192 | maior diferença absoluta: 7.275957614183426e-12


In [17]:
# Data de cada célula: a coluna indice guarda o rótulo da linha na base de onde a célula saiu.
# .values pega só os valores, sem o índice, para não desalinhar com as linhas das células
celulas_tratada["Data"] = df_tratada.loc[celulas_tratada["indice"], "Data"].values
celulas["Data"] = df_med.loc[celulas["indice"], "Data"].values

# merge junta as duas tabelas pela mesma célula
chaves = ["seed", "pct_faltantes", "metodo", "variavel", "Data"]
celulas_pares = celulas_tratada.merge(celulas, on=chaves, suffixes=("_tratada", "_nao_tratada"))
print("Células: tratada", len(celulas_tratada), "| não tratada", len(celulas), "| pares", len(celulas_pares))

# Conferência: o valor verdadeiro é o mesmo nas duas bases em todas as células (precisa ser True)
print("Valor verdadeiro igual em todas as células:",
      (celulas_pares["verdadeiro_tratada"] == celulas_pares["verdadeiro_nao_tratada"]).all())

# DP de cada variável nos 118 meses da prova (o std do pandas usa ddof=1)
dp = df_med.loc[meses_sorteio, variaveis].std()

# igual: True quando o valor imputado é o mesmo nas duas bases
celulas_pares["igual"] = np.isclose(celulas_pares["imputado_tratada"], celulas_pares["imputado_nao_tratada"])

# Mudança em DP: abs() tira o sinal; map troca o nome da variável de cada linha pelo DP dela
mudanca = (celulas_pares["imputado_tratada"] - celulas_pares["imputado_nao_tratada"]).abs()
celulas_pares["mudanca_dp"] = mudanca / celulas_pares["variavel"].map(dp)

Células: tratada 106560 | não tratada 106560 | pares 106560
Valor verdadeiro igual em todas as células: True


Resumo por método e taxa (as 16 variáveis juntas): a % de células com o mesmo valor imputado nas duas bases e a mudança em DP (a média, a mediana e o valor abaixo do qual ficam 90% das células).

In [18]:
linhas = []

for metodo in metodos:
    for taxa in taxas:
        # Todas as células deste método e desta taxa (16 variáveis × 30 rodadas)
        grupo = celulas_pares[(celulas_pares["metodo"] == metodo) & (celulas_pares["pct_faltantes"] == taxa)]

        linhas.append({
            "metodo": metodo,
            "pct_faltantes": taxa,
            "n_celulas": len(grupo),
            "pct_iguais": 100 * grupo["igual"].mean(),    # a média de True/False é a fração de True
            "mudanca_media_dp": grupo["mudanca_dp"].mean(),
            "mudanca_mediana_dp": grupo["mudanca_dp"].median(),
            "mudanca_p90_dp": grupo["mudanca_dp"].quantile(0.9)    # 90% das células mudam menos que isso
        })

pd.DataFrame(linhas).round(3)

,metodo,pct_faltantes,n_celulas,pct_iguais,mudanca_media_dp,mudanca_mediana_dp,mudanca_p90_dp
0,media,0.05,2400,0.000,0.099,0.091,0.166
1,media,0.10,5280,0.000,0.104,0.096,0.175
2,media,0.20,11040,0.000,0.116,0.104,0.194
3,media,0.30,16800,0.000,0.131,0.120,0.221
4,mediana,0.05,2400,40.833,0.034,0.014,0.084
5,mediana,0.10,5280,39.375,0.042,0.016,0.086
6,mediana,0.20,11040,39.167,0.047,0.016,0.120
7,mediana,0.30,16800,35.833,0.053,0.022,0.120
8,knn,0.05,2400,3.625,0.073,0.040,0.178
9,knn,0.10,5280,3.030,0.077,0.044,0.188


Por variável: uma tabela com a % de células iguais e outra com a mudança média em DP; variáveis nas linhas; uma coluna para cada método e taxa.

In [19]:
linhas = []

for metodo in metodos:
    for taxa in taxas:
        do_metodo_taxa = celulas_pares[(celulas_pares["metodo"] == metodo) & (celulas_pares["pct_faltantes"] == taxa)]

        # Cada grupo = as células de uma variável (30 rodadas)
        for variavel, grupo in do_metodo_taxa.groupby("variavel", sort=False):
            linhas.append({
                "metodo": metodo,
                "pct_faltantes": taxa,
                "variavel": variavel,
                "n_celulas": len(grupo),
                "pct_iguais": 100 * grupo["igual"].mean(),
                "mudanca_media_dp": grupo["mudanca_dp"].mean(),
                "mudanca_mediana_dp": grupo["mudanca_dp"].median(),
                "mudanca_p90_dp": grupo["mudanca_dp"].quantile(0.9)
            })

celulas_que_mudam = pd.DataFrame(linhas)

for coluna in ["pct_iguais", "mudanca_media_dp"]:
    # pivot: variáveis nas linhas; colunas em dois níveis (método, taxa)
    tabela_coluna = celulas_que_mudam.pivot(index="variavel", columns=["metodo", "pct_faltantes"], values=coluna)

    # Variáveis na ordem da base e métodos na ordem média, mediana, KNN
    tabela_coluna = tabela_coluna.loc[variaveis, metodos]

    print(coluna)
    display(tabela_coluna.round(3))

pct_iguais


metodo        media                 mediana                               knn  \
pct_faltantes  0.05 0.10 0.20 0.30     0.05     0.10     0.20     0.30   0.05   
variavel                                                                        
COR             0.0  0.0  0.0  0.0    0.000    0.000    3.333    0.000  2.000   
TURB.           0.0  0.0  0.0  0.0   10.000   26.667   23.333   33.333  2.667   
pH              0.0  0.0  0.0  0.0   86.667   73.333   66.667   70.000  4.000   
ALC.            0.0  0.0  0.0  0.0   86.667   70.000   66.667   56.667  4.000   
AC.             0.0  0.0  0.0  0.0  100.000  100.000  100.000  100.000  6.000   
O.C.            0.0  0.0  0.0  0.0    0.000    0.000    0.000    0.000  3.333   
O.D.            0.0  0.0  0.0  0.0  100.000   76.667   83.333   66.667  5.333   
Cl              0.0  0.0  0.0  0.0  100.000   96.667   93.333   76.667  2.667   
DUR.            0.0  0.0  0.0  0.0    0.000    0.000    3.333    6.667  4.000   
Fe              0.0  0.0  0.0  0.0   23.333   16.667   33.333   20.000  1.333   
Mn              0.0  0.0  0.0  0.0   96.667   96.667   63.333   56.667  3.333   
Cond.           0.0  0.0  0.0  0.0    0.000    0.000    0.000    0.000  2.000   
Cianobacteria   0.0  0.0  0.0  0.0    0.000    0.000    0.000    0.000  3.333   
C.F.            0.0  0.0  0.0  0.0    0.000   10.000   23.333   23.333  4.667   
Clorofila       0.0  0.0  0.0  0.0    0.000    0.000    0.000   13.333  6.667   
F               0.0  0.0  0.0  0.0   50.000   63.333   66.667   50.000  2.667   

metodo                              
pct_faltantes   0.10   0.20   0.30  
variavel                            
COR            2.121  2.319  2.095  
TURB.          2.424  2.609  2.381  
pH             4.242  4.058  2.952  
ALC.           3.333  3.043  3.048  
AC.            3.636  2.899  3.048  
O.C.           2.727  3.043  2.190  
O.D.           4.242  4.058  3.048  
Cl             3.030  2.174  2.190  
DUR.           2.121  2.464  2.000  
Fe             2.121  1.884  2.952  
Mn             1.818  2.464  2.381  
Cond.          2.424  2.319  2.286  
Cianobacteria  3.030  2.899  3.143  
C.F.           3.939  2.609  2.857  
Clorofila      4.242  2.464  2.762  
F              3.030  2.464  2.476

mudanca_media_dp


metodo         media                      mediana                         knn  \
pct_faltantes   0.05   0.10   0.20   0.30    0.05   0.10   0.20   0.30   0.05   
variavel                                                                        
COR            0.066  0.069  0.077  0.088   0.020  0.025  0.024  0.028  0.058   
TURB.          0.042  0.044  0.049  0.057   0.013  0.013  0.015  0.030  0.064   
pH             0.078  0.082  0.092  0.104   0.088  0.175  0.219  0.197  0.094   
ALC.           0.139  0.146  0.164  0.185   0.007  0.017  0.018  0.028  0.060   
AC.            0.128  0.135  0.151  0.171   0.000  0.000  0.000  0.000  0.093   
O.C.           0.116  0.122  0.137  0.155   0.080  0.076  0.084  0.087  0.102   
O.D.           0.010  0.011  0.012  0.014   0.000  0.028  0.020  0.039  0.077   
Cl             0.306  0.322  0.359  0.407   0.000  0.002  0.010  0.025  0.054   
DUR.           0.149  0.156  0.174  0.197   0.146  0.143  0.123  0.130  0.067   
Fe             0.051  0.053  0.060  0.067   0.023  0.029  0.022  0.035  0.065   
Mn             0.109  0.115  0.128  0.145   0.004  0.004  0.044  0.052  0.076   
Cond.          0.101  0.106  0.119  0.135   0.033  0.039  0.057  0.064  0.069   
Cianobacteria  0.026  0.027  0.030  0.035   0.015  0.013  0.014  0.016  0.068   
C.F.           0.014  0.015  0.017  0.018   0.015  0.016  0.012  0.011  0.054   
Clorofila      0.166  0.175  0.195  0.222   0.059  0.058  0.063  0.059  0.088   
F              0.077  0.081  0.090  0.101   0.043  0.032  0.029  0.043  0.074   

metodo                              
pct_faltantes   0.10   0.20   0.30  
variavel                            
COR            0.058  0.059  0.065  
TURB.          0.058  0.070  0.073  
pH             0.087  0.088  0.088  
ALC.           0.064  0.074  0.075  
AC.            0.094  0.096  0.096  
O.C.           0.085  0.107  0.099  
O.D.           0.084  0.080  0.084  
Cl             0.066  0.070  0.074  
DUR.           0.079  0.078  0.082  
Fe             0.062  0.067  0.067  
Mn             0.085  0.083  0.081  
Cond.          0.087  0.081  0.078  
Cianobacteria  0.074  0.072  0.075  
C.F.           0.062  0.072  0.073  
Clorofila      0.103  0.100  0.100  
F              0.080  0.084  0.082

# Gravação
Abas `escolha_k` (a grade na não tratada), `resumo` e `rodadas` (como no notebook da tratada), `comparacao` (uma linha por taxa × métrica × método × variável), `rodadas_ganhas` e `celulas_que_mudam` (uma linha por método × taxa × variável). Rodar de novo dá os mesmos números: as seeds são fixas, e o KNN não tem sorteio.

In [20]:
caminho = BASE_DIR / "data" / "output" / "imputacoes_mesma_prova_nao_tratados.xlsx"

# ExcelWriter permite gravar várias abas no mesmo arquivo
with pd.ExcelWriter(caminho) as arquivo:
    escolha_k.to_excel(arquivo, sheet_name="escolha_k", index=False)
    resumo.to_excel(arquivo, sheet_name="resumo", index=False)
    tabela.to_excel(arquivo, sheet_name="rodadas", index=False)
    comparacao.to_excel(arquivo, sheet_name="comparacao", index=False)
    rodadas_ganhas.to_excel(arquivo, sheet_name="rodadas_ganhas", index=False)
    celulas_que_mudam.to_excel(arquivo, sheet_name="celulas_que_mudam", index=False)

print("Gravado em:", caminho)

Gravado em: C:\_Master\Programacao\Guilherme-Betta\1_Projects\IC2\IC2-item-5\data\output\imputacoes_mesma_prova_nao_tratados.xlsx
